In [4]:
import os
import time
from openai import OpenAI
from tqdm.auto import tqdm

/Users/senudaliyanage/miniconda3/envs/IRP/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
MODEL = "gpt-5-mini"
INPUT_BASE = "data/chunked_test_data"
OUTPUT_BASE = "results/openai_results/openai_summaries_test_chunked"
DELAY_SECONDS = 1
MAX_RETRIES = 3

client = OpenAI(api_key=OPEN_API_KEY)

### System Prompt

In [ ]:
SYSTEM_PROMPT = """
You are a legal research assistant specialized in summarizing legislative documents.
The included text is part of a larger summary.

Create accurate, concise, and short summaries in plain English.
Preserve key legal clauses, definitions, obligations, and relationships.
Do not introduce new information.
Avoid unnecessary simplification.
Maintain legal terminology where important.
"""

### Summarization Function

In [ ]:
def summarize_text(text):
    for attempt in range(MAX_RETRIES):
        try:
            response = client.responses.create(
                model=MODEL,
                max_output_tokens=1024,
                input=[
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": f"Summarize the following legal text:\n\n{text}"
                    }
                ]
            )

            return response.output_text  

        except Exception as e:
            print(f"Retry {attempt+1} due to error: {e}")
            time.sleep(2)

    return None

### Processing Function

In [ ]:
def process_category():
    input_folder = os.path.join(INPUT_BASE)
    output_folder = os.path.join(OUTPUT_BASE)

    os.makedirs(output_folder, exist_ok=True)

    files = [f for f in os.listdir(input_folder) if f.endswith(".txt")]

    for filename in tqdm(files, desc=f"Processing"):

        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, filename)

        # Skip already processed files
        if os.path.exists(output_path):
            continue

        with open(input_path, "r", encoding="utf-8") as f:
            text = f.read()

        summary = summarize_text(text)

        if summary:
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(summary)

        time.sleep(DELAY_SECONDS)

In [13]:
process_category()

Processing: 100%|██████████| 1591/1591 [7:30:09<00:00, 16.98s/it]  
